Install Dependencies

In [1]:
!pip install transformers datasets sentencepiece peft scikit-learn -q


Load data

# Dataset Loading and Filtering


In [2]:
from datasets import load_dataset
print("Loading dataset...")
dataset = load_dataset("lightonai/SwissProt-EC-leaf")
print(dataset)
print("\nFirst example:")
print(dataset["train"][0])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 178302
    })
    test: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 22183
    })
    dev: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 23010
    })
})

First example:
{'seq': 'MKFSEQWLRGWVSPQVDRDALVARLSMAGLEVDSVTPAAGVFSGVVVGEVLSTEQHPDADKLRVCQVSNGAETFQVVCGAPNVRPGLKIPFAMIGAELPGDFKIKKAKLRGVESNGMLCSQAELQIGEGNDGLMELPADASVGEDFRVYLDLEDASIEVDLTPNRGDCLSLAGLAREVGALYDAPVTRPVVMAVPAAHDEVRSVEVLAPAACPRYLGRVIRNVDLSRPTPLWMVERLRRAEVRSIDAAVDITNYVMLELGQPLHAFDLAEINGGIRVRMAEEGEKLVLLDGQEVSLRSDTLVVADHTRALAIAGVMGGEHSGVSATTRDVFLESAFFDQIAVAGKARSYGLHTDASHRYERGVDWQLAREAMERATGLLLEITGGEAGPIIETVSEQHLPSIAPITLRAQRITQMLGMEMDSAEVERLLNALGLKVSADGAGQWRVEVPSHRFDISLEVDLIEELARLYGYNRLPVRYPQARLAPQAKAEARSDLPELRRLLVARGYQEAITYSFIDPKQFELFNPGVEPLLLANPISNDMAAMRSSLWPGLVKALQHNLNRQQDRVRLFESGLRFVGQLEGLKQEPMIAGVVCGSRLPEGWAQGRDTVDFFDVKADVEAVLGFAGALDQFTF

# Dataset Loading and Filtering


In [3]:
## FINAL BENCHMARK: Top-20 EC Classes

from collections import Counter
from datasets import load_dataset
# Reload clean data
dataset = load_dataset("lightonai/SwissProt-EC-leaf")

# Take first EC label only
def first_label(example):
    example["label"] = example["labels"][0]
    return example

train_all = dataset["train"].map(first_label)
val_all = dataset["dev"].map(first_label)
test_all = dataset["test"].map(first_label)

# Find top 20 most common labels in train
counts = Counter(train_all["label"])
top20 = [label for label, count in counts.most_common(20)]
print("Top 20 labels:", top20)

# Keep only top 20 labels
train_data = train_all.filter(lambda x: x["label"] in top20)
val_data = val_all.filter(lambda x: x["label"] in top20)
test_data = test_all.filter(lambda x: x["label"] in top20)

# Limit size so it runs fast
train_data = train_data.shuffle(seed=42).select(range(min(5000, len(train_data))))
val_data = val_data.shuffle(seed=42).select(range(min(1000, len(val_data))))
test_data = test_data.shuffle(seed=42).select(range(min(1000, len(test_data))))

# Remap labels to 0-19
label2id = {label: i for i, label in enumerate(top20)}

def remap(example):
    example["label"] = label2id[example["label"]]
    return example

train_data = train_data.map(remap)
val_data = val_data.map(remap)
test_data = test_data.map(remap)

num_labels = 20

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))
print("Num labels:", num_labels)

Top 20 labels: [3315, 4518, 3746, 4521, 2302, 492, 1613, 2743, 3971, 4514, 309, 3695, 662, 2042, 871, 3982, 4088, 3174, 2084, 3901]


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5000
Val: 1000
Test: 1000
Num labels: 20


# ESM-2 Loading and Tokenization + Linear Probing (2 Epochs)




In [4]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["seq"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_tok = train_data.map(tokenize, batched=True)
val_tok = val_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

remove_cols = ["seq", "labels", "labels_str", "id"]
train_tok = train_tok.remove_columns(remove_cols)
val_tok = val_tok.remove_columns(remove_cols)
test_tok = test_tok.remove_columns(remove_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=20
)

# LINEAR PROBE: freeze ESM2; train only classifier
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

args = TrainingArguments(
    output_dir="./esm2_top20_linear_probe",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
linear_runtime = time.time() - start

linear_test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 TOP-20 LINEAR PROBE RESULTS")
print("==============================")
print(linear_test_results)
print("Runtime seconds:", linear_runtime)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.998597,0.906234,0.760000,0.749744
2,0.740453,0.708634,0.822000,0.825692


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 TOP-20 LINEAR PROBE RESULTS
{'eval_loss': 0.7626010775566101, 'eval_accuracy': 0.813, 'eval_macro_f1': 0.8023283105358123, 'eval_runtime': 2.5484, 'eval_samples_per_second': 392.405, 'eval_steps_per_second': 24.722, 'epoch': 2.0}
Runtime seconds: 31.156949996948242


# Full Fine-Tuning (2 Epochs)

In [5]:
import time
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=20
)

# FULL FINE-TUNING: do not freeze any parameters

args = TrainingArguments(
    output_dir="./esm2_top20_full_finetune",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
full_runtime = time.time() - start

full_test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 TOP-20 FULL FINE-TUNE RESULTS")
print("==============================")
print(full_test_results)
print("Runtime seconds:", full_runtime)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.204675,2.124002,0.752000,0.722656
2,1.881131,1.884357,0.818000,0.817172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 TOP-20 FULL FINE-TUNE RESULTS
{'eval_loss': 1.8864243030548096, 'eval_accuracy': 0.845, 'eval_macro_f1': 0.834525775704177, 'eval_runtime': 2.6891, 'eval_samples_per_second': 371.87, 'eval_steps_per_second': 23.428, 'epoch': 2.0}
Runtime seconds: 93.3458936214447
